# Batch nucleus segmentation for INDI production

## Experimental metadata extraction

### Metadata from epxerimental design

In [226]:
# load in metadata files
import pandas as pd

automated_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_automated_plates_combined.csv")
manual_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_manual_plates_combined.csv")
plate_id = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv")

### Metadata from Opera Phenix

In [227]:
# Define relevant dictionaries for metatdata

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define pseudocolor mapping rules - helpful to generate images on the fly
pseudocolor_map = {
    "DAPI": "blue",
    "Brightfield": "gray",
    "Alexa 488": "green",
    "Alexa 568": "red",
    "Alexa 647": "magenta"
}

# Define pseudocolor mapping rules for matplotlib - helpful to generate images on the fly
mpl_colormaps = {
    "blue": LinearSegmentedColormap.from_list("black_blue", [(0, 0, 0), (0, 0, 1)]),
    "green": LinearSegmentedColormap.from_list("black_green", [(0, 0, 0), (0, 1, 0)]),
    "red": LinearSegmentedColormap.from_list("black_red", [(0, 0, 0), (1, 0, 0)]),
    "magenta": LinearSegmentedColormap.from_list("black_magenta", [(0, 0, 0), (1, 0, 1)]),
    "gray": LinearSegmentedColormap.from_list("black_gray", [(0, 0, 0), (1, 1, 1)])
}

In [228]:
from pathlib import Path
import xml.etree.ElementTree as ET

experiment_name = "b96883e9-18b4-4f0d-ba6e-0ddd08c5413f"

base_path = Path("/data/CARDPB2/iNDI/Production")
experiment_path = base_path / experiment_name

experiment_xml_file = next(experiment_path.glob("*.xml"), None)

experiment_img_dir = experiment_path / "images"

index_xml = next((experiment_path / "index").glob("*.xml"), None)

In [229]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

tree = ET.parse(experiment_xml_file)
OF_dir = Path(experiment_img_dir)
index_tree = ET.parse(index_xml)

root = tree.getroot()

# Namespace mapping
ns = {'h': '43B2A954-E3C3-47E1-B392-6635266B0DD3/HarmonyV7'} # I believe this is consistent between all experiments

# Find elements
measurement_id = root.find('h:MeasurementID', ns).text
date = root.find('h:Date', ns).text
plate = root.find('h:InitialPlateName', ns).text

print("Measurement ID:", measurement_id)
print("Date:", date)
print("Plate:", plate)

root = index_tree.getroot()

# Find elements
plate_id = root.find('.//h:PlateID', ns).text
x_res = float(root.find('.//h:ImageResolutionX', ns).text) * 1e6
y_res = float(root.find('.//h:ImageResolutionY', ns).text) * 1e6

print("Plate:", plate_id)
print(f"X resolution: {x_res} µm")
print(f"Y resolution: {y_res} µm")

channels = []

# Find the Map element that contains channel info entries
for map_el in root.findall(".//h:Map", ns):
    # Check first entry if it has a ChannelName child
    first_entry = map_el.find("h:Entry", ns)
    if first_entry is not None and first_entry.find("h:ChannelName", ns) is not None:
        # Found the Map with channel entries
        for entry in map_el.findall("h:Entry", ns):
            ch_id = entry.attrib.get("ChannelID")
            ch_name = entry.find("h:ChannelName", ns).text
            ch_type = entry.find("h:ChannelType", ns).text if entry.find("h:ChannelType", ns) is not None else None
            excitation = entry.find("h:MainExcitationWavelength", ns).text if entry.find("h:MainExcitationWavelength", ns) is not None else None
            emission = entry.find("h:MainEmissionWavelength", ns).text if entry.find("h:MainEmissionWavelength", ns) is not None else None

            channels.append({
                "ChannelID": int(ch_id) if ch_id is not None else None,
                "Channel_name": ch_name,
                "Type": ch_type,
                "Excitation_nm": excitation,
                "Emission_nm": emission
            })
        break  

# Convert to DataFrame
channel_df = pd.DataFrame(channels).sort_values('ChannelID').reset_index(drop=True)
print(channel_df)

channel_df["Pseudocolor"] = channel_df["Channel_name"].map(pseudocolor_map).fillna("gray")
channel_df["MPL_colormap"] = channel_df["Pseudocolor"].str.lower().map(mpl_colormaps)
channel_df["Measurement_ID"] = measurement_id
channel_df["Measurement_date"] = date
channel_df["Plate_ID"] = plate_id
channel_df["res_x"] = x_res
channel_df["res_y"] = y_res

print(channel_df)

Measurement ID: b96883e9-18b4-4f0d-ba6e-0ddd08c5413f
Date: 2026-02-15T23:02:03.4213063-05:00
Plate: 260216_035603-V
Plate: INDIEL002
X resolution: 0.09418670872212731 µm
Y resolution: 0.09418670872212731 µm
   ChannelID Channel_name          Type Excitation_nm Emission_nm
0          1         DAPI  Fluorescence           375         456
1          2    Alexa 647  Fluorescence           640         706
2          3    Alexa 488  Fluorescence           488         522
3          4    Alexa 568  Fluorescence           561         599
   ChannelID Channel_name          Type Excitation_nm Emission_nm Pseudocolor  \
0          1         DAPI  Fluorescence           375         456        blue   
1          2    Alexa 647  Fluorescence           640         706     magenta   
2          3    Alexa 488  Fluorescence           488         522       green   
3          4    Alexa 568  Fluorescence           561         599         red   

                                        MPL_colormap  \
0

In [230]:
import re
import pandas as pd

# Collect all .tiff files
files = sorted([f for f in OF_dir.rglob('*') if f.suffix.lower() in ['.tiff']])

# Define a function to parse the filename
def parse_filename(name):
    match = re.match(r"r(\d+)c(\d+)f(\d+)p(\d+)-ch(\d+)t(\d+)", name)
    if match:
        return [int(g) for g in match.groups()]
    else:
        return [None] * 6  # Fallback if filename doesn't match

# Create a DataFrame with full path and relative subdirectory
df = pd.DataFrame({
    'filepath': files,
    'filename': [f.name for f in files], 
    'subdirectory': [f.parent.relative_to(OF_dir) for f in files]
})

# Extract metadata from filenames
df[['Row', 'Column', 'Frame', 'Plane', 'ChannelID', 'Time']] = df['filename'].apply(
    lambda x: pd.Series(parse_filename(x))
)

print(df.head())

                                            filepath  \
0  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
1  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
2  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
3  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
4  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   

                    filename subdirectory  Row  Column  Frame  Plane  \
0  r02c03f01p01-ch01t01.tiff       r02c03    2       3      1      1   
1  r02c03f01p01-ch02t01.tiff       r02c03    2       3      1      1   
2  r02c03f01p01-ch03t01.tiff       r02c03    2       3      1      1   
3  r02c03f01p01-ch04t01.tiff       r02c03    2       3      1      1   
4  r02c03f02p01-ch01t01.tiff       r02c03    2       3      2      1   

   ChannelID  Time  
0          1     1  
1          2     1  
2          3     1  
3          4     1  
4          1     1  


In [231]:
merged_ChannelID_df = pd.merge(df, channel_df, on="ChannelID")
print(merged_ChannelID_df)

                                                filepath  \
0      /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
1      /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
2      /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
3      /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
4      /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
...                                                  ...   
41155  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
41156  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
41157  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
41158  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   
41159  /data/CARDPB2/iNDI/Production/b96883e9-18b4-4f...   

                        filename subdirectory  Row  Column  Frame  Plane  \
0      r02c03f01p01-ch01t01.tiff       r02c03    2       3      1      1   
1      r02c03f01p01-ch02t01.tiff       r02c03    2       3      1      1   
2      r02c03f01p01-ch03t01.tiff       r02c03    2 

In [232]:
summary = {
    "wells": merged_ChannelID_df[['Row', 'Column']].drop_duplicates().shape[0],
    "channels": merged_ChannelID_df['ChannelID'].nunique(),
    "z_planes": merged_ChannelID_df['Plane'].nunique(),
    "frames": merged_ChannelID_df['Frame'].nunique(),
    "timepoints": merged_ChannelID_df['Time'].nunique(),
}

print(summary)

print(f"""
Experiment ID: {measurement_id}
Plate ID: {plate}
Wells imaged: {summary["wells"]}
Frames per well: {summary["frames"]}
Channels per image: {summary["channels"]}
Z-slices per image: {summary["z_planes"]}
Timepoints per image: {summary["timepoints"]}
""")

{'wells': 294, 'channels': 4, 'z_planes': 1, 'frames': 35, 'timepoints': 1}

Experiment ID: b96883e9-18b4-4f0d-ba6e-0ddd08c5413f
Plate ID: 260216_035603-V
Wells imaged: 294
Frames per well: 35
Channels per image: 4
Z-slices per image: 1
Timepoints per image: 1



In [233]:
import dask
from dask import delayed, compute
import pandas as pd
import tifffile
import numpy as np
from skimage import filters, morphology
from scipy import ndimage as ndi
from skimage.measure import regionprops_table
from scipy.stats import norm
import os

# --- Parameters ---
intensity_scaling_param = [1, 7]
blur_sigma = 1
min_area = 3500

# --- Function to process a single image ---
@delayed
def process_nucleus_image(image_path):
    # Load image
    nuc = tifffile.imread(image_path)
    
    # Normalize
    m, s = norm.fit(nuc.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, nuc.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, nuc.max())
    nuc_stretch = np.clip(nuc, stretch_min, stretch_max)
    image_norm = (nuc_stretch - stretch_min) / (stretch_max - stretch_min)

    # Blur
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # Step 1: Low level threshold
    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2
    img_low_level = blurred > th_low_cutoff
    img_low_level = morphology.remove_small_objects(img_low_level.astype(bool), min_size=min_area)
    img_low_level = morphology.dilation(img_low_level, footprint=morphology.disk(2))

    # Step 2: High level threshold
    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level)
    lab_low, num_obj = morphology.label(img_low_level, return_num=True)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        local_otsu = filters.threshold_otsu(blurred[single_obj])
        if local_otsu > otsu_cutoff:
            img_high_level[np.logical_and(blurred > 0.98 * local_otsu, single_obj)] = 1

    filled = ndi.binary_fill_holes(img_high_level)
    filled = morphology.dilation(filled, footprint=morphology.disk(2))
    labeled_filled = morphology.label(filled)
    nuc_seg = morphology.remove_small_objects(labeled_filled, min_size=min_area)

    # Extract features
    props = regionprops_table(
        label_image=nuc_seg,
        intensity_image=nuc,
        properties=(
            'label', 'area', 'mean_intensity', 'max_intensity', 'min_intensity', 'std_intensity',
            'centroid', 'eccentricity', 'solidity', 'perimeter', 'feret_diameter_max',
            'orientation', 'major_axis_length', 'minor_axis_length'
        )
    )
    df = pd.DataFrame(props)
    df['image_name'] = os.path.basename(image_path)
    
    return df

In [234]:
# Filter for DAPI images
dapi_samples = merged_ChannelID_df[merged_ChannelID_df['Channel_name'] == "DAPI"].copy() 

# Extract file paths
filepaths = dapi_samples['filepath'].tolist()

# --- Wrap all images with delayed ---
tasks = [process_nucleus_image(fp) for fp in filepaths]

In [235]:
from dask.diagnostics import ProgressBar
from datetime import datetime

today = datetime.now().strftime("%Y%m%d")

out_path = f"{experiment_name}_nuclei_features_test_{today}.csv"

with ProgressBar():
    dfs = dask.compute(*tasks, scheduler="processes")

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv(out_path, index=False)

[                                        ] | 0% Completed | 20.14 sms

/tmp/ipykernel_3824817/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?
/tmp/ipykernel_3824817/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[                                        ] | 0% Completed | 21.05 s

/tmp/ipykernel_3824817/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[                                        ] | 0% Completed | 22.56 s

...

[######################################  ] | 96% Completed | 19m 34s

/tmp/ipykernel_3824817/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[######################################  ] | 97% Completed | 19m 45s

/tmp/ipykernel_3824817/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[########################################] | 100% Completed | 19m 55s


In [236]:
# make an output for all warning files (separate csv)
nuc_csv = f"{experiment_name}_nuclei_features_test_20260317.csv"
nuc_feat = pd.read_csv(nuc_csv)

global_summary = pd.DataFrame({
    "image_count": [nuc_feat["image_name"].nunique()],
    "total_nuclei": [len(nuc_feat)],
    "mean_nuclei_per_image": [len(nuc_feat) / nuc_feat["image_name"].nunique()]
})

print(global_summary)

summary = (
    nuc_feat.groupby("image_name")
    .agg(
        nuclei_count=("label", "count"),
        
        # Area stats
        total_area=("area", "sum"),
        mean_area=("area", "mean"),
        median_area=("area", "median"),
        std_area=("area", "std"),
        
        # Intensity stats
        mean_intensity_mean=("mean_intensity", "mean"),
        mean_intensity_std=("mean_intensity", "std"),
        max_intensity_mean=("max_intensity", "mean"),
        
        # Shape stats
        mean_eccentricity=("eccentricity", "mean"),
        mean_solidity=("solidity", "mean"),
        
        # Size/shape descriptors
        mean_major_axis=("major_axis_length", "mean"),
        mean_minor_axis=("minor_axis_length", "mean"),
        
        # Spatial (optional but useful for QC)
        mean_centroid_y=("centroid-0", "mean"),
        mean_centroid_x=("centroid-1", "mean"),
    )
    .reset_index()
)

print(summary)

   image_count  total_nuclei  mean_nuclei_per_image
0        10217        103534              10.133503
                      image_name  nuclei_count  total_area    mean_area  \
0      r02c03f01p01-ch01t01.tiff            11     75863.0  6896.636364   
1      r02c03f02p01-ch01t01.tiff            15    149800.0  9986.666667   
2      r02c03f03p01-ch01t01.tiff            14     96658.0  6904.142857   
3      r02c03f04p01-ch01t01.tiff            15     93224.0  6214.933333   
4      r02c03f05p01-ch01t01.tiff            15     89948.0  5996.533333   
...                          ...           ...         ...          ...   
10212  r15c23f31p01-ch01t01.tiff             5     30044.0  6008.800000   
10213  r15c23f32p01-ch01t01.tiff            12     86118.0  7176.500000   
10214  r15c23f33p01-ch01t01.tiff            19    116049.0  6107.842105   
10215  r15c23f34p01-ch01t01.tiff             3     18458.0  6152.666667   
10216  r15c23f35p01-ch01t01.tiff            19    125998.0  6631.473684